In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Fix TotalCharges again (fresh start)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(0, inplace=True)

print("Data loaded! Shape:", df.shape)

Data loaded! Shape: (7043, 21)


C:\Users\ishika binage\AppData\Local\Temp\ipykernel_1944\1993318166.py:11: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['TotalCharges'].fillna(0, inplace=True)


In [2]:
df.drop('customerID', axis=1, inplace=True)
print("customerID dropped!")
print("New shape:", df.shape)

customerID dropped!
New shape: (7043, 20)


In [3]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print("Churn column encoded!")
print(df['Churn'].value_counts())

Churn column encoded!
Churn
0    5174
1    1869
Name: count, dtype: int64


In [4]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns:")
print(cat_cols)

Categorical columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\ishika binage\AppData\Local\Temp\ipykernel_1944\761230320.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


In [5]:
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])
    
print("All categorical columns encoded!")
df.head()

All categorical columns encoded!


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


In [6]:
X = df.drop('Churn', axis=1)
y = df['Churn']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nChurn distribution before SMOTE:")
print(y.value_counts())

Features shape: (7043, 19)
Target shape: (7043,)

Churn distribution before SMOTE:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [9]:
# Check NaN before scaling
print("NaN values before fix:")
print(pd.DataFrame(X_scaled).isnull().sum().sum())

# Fill any remaining NaN with 0
X_scaled = np.nan_to_num(X_scaled, nan=0.0)

print("NaN values after fix:", np.isnan(X_scaled).sum())
print("Features scaled and cleaned!")

NaN values before fix:
11
NaN values after fix: 0
Features scaled and cleaned!


In [10]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

print("After SMOTE:")
print("X shape:", X_resampled.shape)
print("y distribution:")
print(pd.Series(y_resampled).value_counts())

After SMOTE:
X shape: (10348, 19)
y distribution:
Churn
0    5174
1    5174
Name: count, dtype: int64


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled,
    test_size=0.2,
    random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (8278, 19)
Test size: (2070, 19)


In [12]:
import joblib

# Save the scaler so we can reuse it later
joblib.dump(scaler, '../models/scaler.pkl')

# Save train/test splits as numpy arrays
np.save('../models/X_train.npy', X_train)
np.save('../models/X_test.npy', X_test)
np.save('../models/y_train.npy', y_train)
np.save('../models/y_test.npy', y_test)

# Save feature names for later use
feature_names = X.columns.tolist()
joblib.dump(feature_names, '../models/feature_names.pkl')

print("All data saved to models/ folder!")

All data saved to models/ folder!


## Preprocessing Summary

- Dropped customerID (not useful for prediction)
- Encoded target column: Yes → 1, No → 0
- Label encoded all 16 categorical columns
- Scaled all features using StandardScaler
- Applied SMOTE to fix class imbalance
- Split data: 80% train, 20% test
- Saved scaler and data splits to models/ folder